In [1]:
import os
from pathlib import Path
import time
import warnings
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_absolute_percentage_error)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Try importing LightGBM and CatBoost
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

try:
    from catboost import CatBoostRegressor
    HAS_CAT = True
except ImportError:
    HAS_CAT = False

warnings.filterwarnings("ignore")

In [2]:
RANDOM_STATE = 42
OUTPUT_DIR   = Path("carbon24_energy_results")
FIGURE_DIR   = OUTPUT_DIR / "figures"
DATA_PATH    = Path("carbon24_pipeline_results/tier3_gmm_labeled.csv")

# Ensure directories exist
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

STRUCT_FEATURES = [
    "num_atoms", "a", "b", "c", "alpha", "beta", "gamma",
    "volume", "volume_per_atom", "b_over_a", "c_over_a", "angle_deviation",
    "mean_bond_length", "std_bond_length", "min_bond_length", "max_bond_length",
    "std_coordination", "min_coordination", "max_coordination",
]

CLUSTER_FEATURES = ["kmeans_cluster", "gmm_cluster", "hdbscan_probability", "pca1", "pca2"]
CAT_FEATURES = ["crystal_system", "space_group_symbol", "scientific_label"]
TARGET = "relative_energy"

In [3]:
def regression_metrics(y_true, y_pred, prefix=""):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mask = np.abs(y_true) > 0.01
    mape = (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100
            if mask.sum() > 0 else np.nan)
    return {
        f"{prefix}RMSE": round(rmse, 6),
        f"{prefix}MAE":  round(mae,  6),
        f"{prefix}R2":   round(r2,   6),
        f"{prefix}MAPE%": round(mape, 4) if not np.isnan(mape) else None,
    }

def load_and_prepare():
    df = pd.read_csv(DATA_PATH)
    le_dict = {}
    for col in CAT_FEATURES:
        if col in df.columns:
            le = LabelEncoder()
            df[f"{col}_enc"] = le.fit_transform(df[col].fillna("Unknown").astype(str))
            le_dict[col] = le

    enc_cats = [f"{c}_enc" for c in CAT_FEATURES if c in df.columns]
    all_features = [f for f in STRUCT_FEATURES if f in df.columns] + \
                   [f for f in CLUSTER_FEATURES if f in df.columns] + enc_cats

    train = df[df["split"] == "train"].copy()
    val   = df[df["split"] == "validation"].copy()
    test  = df[df["split"] == "test"].copy()

    X_train = train[all_features].fillna(train[all_features].median())
    X_val   = val[all_features].fillna(train[all_features].median())
    X_test  = test[all_features].fillna(train[all_features].median())

    return X_train, X_val, X_test, train[TARGET].values, val[TARGET].values, test[TARGET].values, all_features, test

In [4]:
def regression_metrics(y_true, y_pred, prefix=""):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mask = np.abs(y_true) > 0.01
    mape = (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask]).mean() * 100
            if mask.sum() > 0 else np.nan)
    return {
        f"{prefix}RMSE": round(rmse, 6),
        f"{prefix}MAE":  round(mae,  6),
        f"{prefix}R2":   round(r2,   6),
        f"{prefix}MAPE%": round(mape, 4) if not np.isnan(mape) else None,
    }

def load_and_prepare():
    df = pd.read_csv(DATA_PATH)
    le_dict = {}
    for col in CAT_FEATURES:
        if col in df.columns:
            le = LabelEncoder()
            df[f"{col}_enc"] = le.fit_transform(df[col].fillna("Unknown").astype(str))
            le_dict[col] = le

    enc_cats = [f"{c}_enc" for c in CAT_FEATURES if c in df.columns]
    all_features = [f for f in STRUCT_FEATURES if f in df.columns] + \
                   [f for f in CLUSTER_FEATURES if f in df.columns] + enc_cats

    train = df[df["split"] == "train"].copy()
    val   = df[df["split"] == "validation"].copy()
    test  = df[df["split"] == "test"].copy()

    X_train = train[all_features].fillna(train[all_features].median())
    X_val   = val[all_features].fillna(train[all_features].median())
    X_test  = test[all_features].fillna(train[all_features].median())

    return X_train, X_val, X_test, train[TARGET].values, val[TARGET].values, test[TARGET].values, all_features, test

In [5]:
# Gọi hàm để lấy dữ liệu và mô hình
X_train, X_val, X_test, y_train, y_val, y_test, features, test_df = load_and_prepare()
models = get_models()

leaderboard = []
predictions = {}
feature_importances = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    pred_test = model.predict(X_test)
    
    metrics = regression_metrics(y_test, pred_test, "test_")
    metrics.update({"Model": name})
    leaderboard.append(metrics)
    predictions[name] = pred_test

lb_df = pd.DataFrame(leaderboard).sort_values("test_RMSE")
print(lb_df)

NameError: name 'get_models' is not defined

In [ ]:
plot_leaderboard(lb_df)